In [2]:
import netket as nk
import numpy as np
import matplotlib.pyplot as plt
import json
from pyscf import gto, scf, fci
import netket.experimental as nkx
import jax
import jax.numpy as jnp
from flax import nnx
import jax.tree_util as jtu
import sys
sys.path.insert(0, '.')


In [3]:
# H2分子几何构型
bond_length = 1.4  # 平衡键长（埃）
geometry = [
    ('H', (0., 0., 0.)),
    ('H', (bond_length, 0., 0.)),
]

print("="*60)
print("H2分子系统设置")
print("="*60)
print(f"键长: {bond_length} 埃")

# 创建分子对象
mol = gto.M(atom=geometry, basis='STO-3G')

# Hartree-Fock计算
mf = scf.RHF(mol).run(verbose=0)
E_hf = mf.e_tot
print(f"\nHartree-Fock能量: {E_hf:.8f} Ha")

# FCI计算（参考值）
cisolver = fci.FCI(mf)
cisolver.nroots = 4
E_fcis, fcivec = cisolver.kernel()

print(f"\nFCI能级（参考值）:")
print("-"*50)
for i, e in enumerate(E_fcis):
    exc_energy = (e - E_fcis[0]) * 27.2114
    if i == 0:
        print(f"  E{i} (基态)     = {e:.8f} Ha")
    else:
        print(f"  E{i} (第{i}激发态) = {e:.8f} Ha  激发能: {exc_energy:.4f} eV")

# 创建NetKet哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

H2分子系统设置
键长: 1.4 埃

Hartree-Fock能量: -0.94148065 Ha

FCI能级（参考值）:
--------------------------------------------------
  E0 (基态)     = -1.01546825 Ha
  E1 (第1激发态) = -0.87542794 Ha  激发能: 3.8107 eV
  E2 (第2激发态) = -0.42938376 Ha  激发能: 15.9482 eV
  E3 (第3激发态) = -0.26922131 Ha  激发能: 20.3064 eV


In [4]:
# 创建Hilbert空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1, 1)
)

print("="*60)
print("Hilbert空间信息")
print("="*60)
print(f"空间轨道数: 2")
print(f"自旋轨道数: 4")
print(f"电子数: 2 (α=1, β=1)")
print(f"Hilbert空间维度: {hi.n_states}")
print(f"\n所有可能的电子组态:")
print(hi.all_states())

# 创建采样器
g = nk.graph.Graph(edges=[(0,1),(2,3)])
sampler = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=16, spin_symmetric=True, sweep_size=64
)

Hilbert空间信息
空间轨道数: 2
自旋轨道数: 4
电子数: 2 (α=1, β=1)
Hilbert空间维度: 4

所有可能的电子组态:
[[0 1 0 1]
 [0 1 1 0]
 [1 0 0 1]
 [1 0 1 0]]


In [6]:
class FFNN(nnx.Module):
    """前馈神经网络波函数"""
    
    def __init__(self, n_orbitals: int, hidden_dim: int = 8, *, rngs: nnx.Rngs):
        self.n_orbitals = n_orbitals
        self.hidden_dim = hidden_dim
        
        self.linear1 = nnx.Linear(
            in_features=n_orbitals, 
            out_features=hidden_dim, 
            rngs=rngs, 
            param_dtype=complex
        )
        self.linear2 = nnx.Linear(
            in_features=hidden_dim, 
            out_features=hidden_dim, 
            rngs=rngs, 
            param_dtype=complex
        )
        self.output_layer = nnx.Linear(
            in_features=hidden_dim, 
            out_features=1, 
            rngs=rngs, 
            param_dtype=complex
        )
    
    def __call__(self, x: jax.Array):
        h = self.linear1(x)
        h = nnx.tanh(h)
        h = self.linear2(h)
        h = nnx.tanh(h)
        out = self.output_layer(h)
        return jnp.squeeze(out, axis=-1)

n_spin_orbitals = 4
hidden_dim = 2
model = FFNN(n_orbitals=n_spin_orbitals, hidden_dim=hidden_dim, rngs=nnx.Rngs(42))

params = nnx.state(model, nnx.Param)
n_params = sum(leaf.size for leaf in jtu.tree_leaves(params))

print("="*60)
print("神经网络模型信息")
print("="*60)
print(f"输入维度: {n_spin_orbitals}")
print(f"隐藏层维度: {hidden_dim}")
print(f"总参数量: {n_params}")

神经网络模型信息
输入维度: 4
隐藏层维度: 2
总参数量: 19


In [ ]:
import jax
import jax.numpy as jnp

FFNN_1 = FFNN(n_orbitals=4, hidden_dim=2, rngs=nnx.Rngs(40))
FFNN_2 = FFNN(n_orbitals=4, hidden_dim=2, rngs=nnx.Rngs(41))
FFNN_3 = FFNN(n_orbitals=4, hidden_dim=2, rngs=nnx.Rngs(42))
FFNN_4 = FFNN(n_orbitals=4, hidden_dim=2, rngs=nnx.Rngs(43))


def Wavefunction(x: jax.Array):
    return jnp.array([FFNN_1(x),FFNN_2(x),FFNN_3(x),FFNN_4(x)])


In [10]:
states = hi.all_states()

In [16]:
row1 = jnp.array([FFNN_1(states[0]),FFNN_2(states[0]),FFNN_3(states[0]),FFNN_4(states[0])])
row2 = jnp.array([FFNN_1(states[1]),FFNN_2(states[1]),FFNN_3(states[1]),FFNN_4(states[1])])
row3 = jnp.array([FFNN_1(states[2]),FFNN_2(states[2]),FFNN_3(states[2]),FFNN_4(states[2])])
row4 = jnp.array([FFNN_1(states[3]),FFNN_2(states[3]),FFNN_3(states[3]),FFNN_4(states[3])])

In [14]:
FFNN_1(states[0])

Array(0.18923584-0.1891446j, dtype=complex128)

## NES-VMC 核心实现

### 核心思想

根据 Pfau et al. (2024) 论文，如果我们有 $k$ 个线性无关的变分波函数 $\psi_1, \psi_2, ..., \psi_k$，可以构造哈密顿量矩阵：

$$\tilde{H}_{ij} = \langle\psi_i|H|\psi_j\rangle$$

通过最小化 $\text{Tr}(\tilde{H})$ 并对角化 $\tilde{H}$，可以得到各激发态能量。

### 实现策略

1. **构造哈密顿量矩阵** $H_{ij} = \langle\psi_i|H|\psi_j\rangle$
2. **构造重叠矩阵** $S_{ij} = \langle\psi_i|\psi_j\rangle$
3. **求解广义特征值问题** $H c = E S c$
4. **优化神经网络参数** 使对角化后的能量最小化

In [ ]:
import jax
import jax.numpy as jnp
from functools import partial

print("="*60)
print("NES-VMC 核心函数")
print("="*60)

def compute_local_energy(model, params, samples, hamiltonian):
    """计算局部能量 E_L(σ) = Hψ(σ)/ψ(σ)"""
    psi = model.apply(params, samples)
    
    
    H_psi = jnp.zeros_like(psi)
    for i in range(len(samples)):
        sigma = samples[i]
        
        
        H_psi_i = 0.0
        for j in range(hi.n_states):
            sigma_j = states[j]
            H_ij = ha.get_matrix(sigma, sigma_j)
            psi_j = model.apply(params, sigma_j.reshape(1, -1))[0]
            H_psi_i += H_ij * psi_j
        H_psi = H_psi.at[i].set(H_psi_i)
    
    E_local = H_psi / psi
    return E_local

def compute_matrices(models, params_list, samples, hamiltonian):
    """
    计算哈密顿量矩阵和重叠矩阵
    
    参数:
        models: 神经网络模型列表
        params_list: 参数列表
        samples: 采样点
        hamiltonian: 哈密顿量
    
    返回:
        H_matrix: 哈密顿量矩阵
        S_matrix: 重叠矩阵
    """
    n_states = len(models)
    n_samples = len(samples)
    
    H_matrix = jnp.zeros((n_states, n_states), dtype=complex)
    S_matrix = jnp.zeros((n_states, n_states), dtype=complex)
    
    
    psi_matrix = jnp.zeros((n_states, n_samples), dtype=complex)
    for i in range(n_states):
        psi_matrix = psi_matrix.at[i].set(models[i].apply(params_list[i], samples))
    
    
    for i in range(n_states):
        for j in range(n_states):
            
            ratio = jnp.exp(jnp.log(psi_matrix[j]) - jnp.log(psi_matrix[i]))
            S_ij = jnp.mean(ratio)
            S_matrix = S_matrix.at[i, j].set(S_ij)
            
            
            E_local_j = compute_local_energy(models[j], params_list[j], samples, hamiltonian)
            H_ij = jnp.mean(ratio * E_local_j)
            H_matrix = H_matrix.at[i, j].set(H_ij)
    
    return H_matrix, S_matrix

def diagonalize_generalized_eigenvalue(H_matrix, S_matrix):
    """
    求解广义特征值问题: H c = E S c
    
    参数:
        H_matrix: 哈密顿量矩阵
        S_matrix: 重叠矩阵
    
    返回:
        energies: 特征值（能量）
        coefficients: 特征向量（波函数系数）
    """
    
    H_herm = (H_matrix + H_matrix.conj().T) / 2
    S_herm = (S_matrix + S_matrix.conj().T) / 2
    
    
    reg = 1e-6
    S_reg = S_herm + reg * jnp.eye(S_herm.shape[0], dtype=complex)
    
    try:
        
        L = jnp.linalg.cholesky(S_reg)
        L_inv = jnp.linalg.inv(L)
        H_prime = L_inv @ H_herm @ L_inv.conj().T
        
        
        energies, coeffs_prime = jnp.linalg.eigh(H_prime)
        
        
        coefficients = L_inv.conj().T @ coeffs_prime
    except Exception as e:
        
        S_inv = jnp.linalg.inv(S_reg)
        H_eff = S_inv @ H_herm
        H_eff = (H_eff + H_eff.conj().T) / 2
        energies, coefficients = jnp.linalg.eigh(H_eff)
    
    return energies, coefficients

def compute_energy_trace(H_matrix):
    """计算哈密顿量矩阵的迹（总能量）"""
    return jnp.trace(H_matrix)

print("核心函数定义完成！")
print("- compute_matrices: 计算哈密顿量矩阵和重叠矩阵")
print("- diagonalize_generalized_eigenvalue: 求解广义特征值问题")
print("- compute_energy_trace: 计算总能量")

In [ ]:
print("="*60)
print("测试矩阵计算")
print("="*60)

models = [FFNN_1, FFNN_2, FFNN_3, FFNN_4]
params_list = [nnx.state(m, nnx.Param) for m in models]

print(f"\n模型数量: {len(models)}")
print(f"采样点数量: {len(states)}")

H_matrix, S_matrix = compute_matrices(models, params_list, states, ha)

print(f"\n哈密顿量矩阵 H:")
for i in range(len(H_matrix)):
    for j in range(len(H_matrix[i])):
        print(f"  H[{i},{j}] = {H_matrix[i,j]:.6f}", end="  ")
    print()

print(f"\n重叠矩阵 S:")
for i in range(len(S_matrix)):
    for j in range(len(S_matrix[i])):
        print(f"  S[{i},{j}] = {S_matrix[i,j]:.6f}", end="  ")
    print()

energies, coeffs = diagonalize_generalized_eigenvalue(H_matrix, S_matrix)

print(f"\n对角化后的能量:")
for i, e in enumerate(energies):
    print(f"  E{i} = {e.real:.8f} Ha")

print(f"\nFCI参考能量:")
for i, e in enumerate(E_fcis):
    print(f"  E{i} = {e:.8f} Ha")

errors = [abs(energies[i].real - E_fcis[i]) for i in range(len(energies))]
print(f"\n误差:")
for i, err in enumerate(errors):
    print(f"  ΔE{i} = {err:.6f} Ha ({err*27.2114*1000:.2f} meV)")

In [ ]:
print("="*60)
print("计算能量对参数的梯度")
print("="*60)

@partial(jax.jit)
def energy_loss(params_list, models, samples, hamiltonian):
    """
    计算总能量损失函数
    
    L = Tr(H) = Σ_i ⟨ψ_i|H|ψ_i⟩
    """
    H_matrix, _ = compute_matrices(models, params_list, samples, hamiltonian)
    return jnp.trace(H_matrix)

@partial(jax.jit)
def compute_gradients(params_list, models, samples, hamiltonian):
    """计算所有模型参数的梯度"""
    loss_fn = lambda *params: energy_loss(list(params), models, samples, hamiltonian)
    
    grad_fn = jax.grad(loss_fn)
    grads = grad_fn(*params_list)
    
    return grads

print("测试梯度计算...")

grads = compute_gradients(params_list, models, states, ha)

print(f"\n梯度计算完成！")
print(f"模型数量: {len(grads)}")

for i, grad in enumerate(grads):
    total_grad = sum(jnp.sum(jnp.abs(v)) for v in jax.tree_util.tree_leaves(grad))
    print(f"  模型 {i} 的梯度范数: {total_grad:.6f}")

In [ ]:
print("="*60)
print("NES-VMC 优化循环")
print("="*60)

import optax

n_states = 4
n_iter = 100
learning_rate = 0.01

models = [FFNN_1, FFNN_2, FFNN_3, FFNN_4]
params_list = [nnx.state(m, nnx.Param) for m in models]

optimizer = optax.adam(learning_rate)
opt_states = [optimizer.init(p) for p in params_list]

print(f"\n优化设置:")
print(f"  模型数量: {n_states}")
print(f"  迭代次数: {n_iter}")
print(f"  学习率: {learning_rate}")
print(f"  优化器: Adam")

energy_history = []
energies_history = []

print(f"\n开始优化...")
for step in range(n_iter):
    
    grads = compute_gradients(params_list, models, states, ha)
    
    
    for i in range(n_states):
        updates, opt_states[i] = optimizer.update(grads[i], opt_states[i])
        params_list[i] = optax.apply_updates(params_list[i], updates)
    
    
    H_matrix, S_matrix = compute_matrices(models, params_list, states, ha)
    energies, coeffs = diagonalize_generalized_eigenvalue(H_matrix, S_matrix)
    
    energy_history.append(jnp.trace(H_matrix).real)
    energies_history.append([e.real for e in energies])
    
    if step % 10 == 0:
        print(f"Step {step:3d}: Tr(H) = {energy_history[-1]:.8f} Ha")
        print(f"         Energies = {[f'{e:.6f}' for e in energies_history[-1]]}")

print(f"\n优化完成！")

print(f"\n最终结果:")
print(f"="*60)
print(f"{'态':<6} {'VMC能量':<15} {'FCI能量':<15} {'误差':<15} {'误差(meV)':<15}")
print("-"*60)
for i in range(n_states):
    vmc_e = energies_history[-1][i]
    fci_e = E_fcis[i]
    error = abs(vmc_e - fci_e)
    error_meV = error * 27.2114 * 1000
    print(f"E{i:<4} {vmc_e:<15.8f} {fci_e:<15.8f} {error:<15.6f} {error_meV:<15.2f}")

In [ ]:
import matplotlib.pyplot as plt

print("="*60)
print("结果可视化")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.plot(energy_history, 'b-', linewidth=2, label='Tr(H)')
ax1.axhline(y=sum(E_fcis), color='r', linestyle='--', linewidth=2, label='FCI Tr(H)')
ax1.set_xlabel('Iteration', fontsize=12)
ax1.set_ylabel('Energy (Ha)', fontsize=12)
ax1.set_title('Total Energy Trace', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

ax2 = axes[1]
energies_array = np.array(energies_history).T
colors = ['r', 'g', 'b', 'orange']
labels = [f'E{i} (VMC)' for i in range(n_states)]
fci_lines = [E_fcis[i] for i in range(n_states)]

for i in range(n_states):
    ax2.plot(energies_array[i], color=colors[i], linewidth=2, label=labels[i])
    ax2.axhline(y=fci_lines[i], color=colors[i], linestyle='--', 
               linewidth=2, alpha=0.5, label=f'E{i} (FCI)')

ax2.set_xlabel('Iteration', fontsize=12)
ax2.set_ylabel('Energy (Ha)', fontsize=12)
ax2.set_title('Individual State Energies', fontsize=14, fontweight='bold')
ax2.legend(fontsize=9, ncol=2)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('nes_vmc_convergence.png', dpi=300, bbox_inches='tight')
plt.show()

print("图表已保存为 'nes_vmc_convergence.png'")

In [ ]:
print("="*60)
print("误差分析")
print("="*60)

final_energies = energies_history[-1]

print(f"\n{'态':<10} {'VMC能量':<18} {'FCI能量':<18} {'绝对误差':<18} {'相对误差':<18}")
print("-"*80)
for i in range(n_states):
    vmc_e = final_energies[i]
    fci_e = E_fcis[i]
    abs_error = abs(vmc_e - fci_e)
    rel_error = abs_error / abs(fci_e) * 100
    abs_error_meV = abs_error * 27.2114 * 1000
    print(f"E{i:<8} {vmc_e:<18.8f} {fci_e:<18.8f} {abs_error:<18.6f} {rel_error:<18.4f}%")
    print(f"{'':10} {'':18} {'':18} {'(meV)':<18} {abs_error_meV:<18.2f}")

print(f"\n激发能对比:")
print(f"{'激发态':<10} {'VMC激发能':<18} {'FCI激发能':<18} {'误差':<18}")
print("-"*60)
for i in range(1, n_states):
    vmc_exc = final_energies[i] - final_energies[0]
    fci_exc = E_fcis[i] - E_fcis[0]
    error = abs(vmc_exc - fci_exc)
    error_eV = error * 27.2114
    print(f"E{i}-E0:<6} {vmc_exc:<18.8f} {fci_exc:<18.8f} {error:<18.6f}")
    print(f"{'':10} {(vmc_exc*27.2114):<18.4f} eV {(fci_exc*27.2114):<18.4f} eV {error_eV:<18.4f} eV")

print(f"\n总结:")
print(f"  - 最大绝对误差: {max(abs(final_energies[i] - E_fcis[i]) for i in range(n_states)):.6f} Ha")
print(f"  - 最大相对误差: {max(abs(final_energies[i] - E_fcis[i]) / abs(E_fcis[i]) * 100 for i in range(n_states)):.4f}%")
print(f"  - 平均绝对误差: {np.mean([abs(final_energies[i] - E_fcis[i]) for i in range(n_states)]):.6f} Ha")